# DataPilot AI — Data Analysis, ML & AI Assistant
**Workflow:** load CSV/Excel → inspect and clean → EDA → train a classification/regression model → export model → use it in Streamlit.

> Run cells from top to bottom. Update `DATA_PATH` and `TARGET_COLUMN` in the indicated cells. This notebook is designed for tabular datasets. Do not use personal or confidential data in a public repository.

## 1. Install dependencies
If packages are missing, run the next cell once, then restart the kernel if Jupyter asks.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn plotly scikit-learn joblib openpyxl

## 2. Import libraries and set project paths

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)

PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
print("Project directory:", PROJECT_DIR.resolve())

## 3. Load your dataset
Change the path to your CSV or Excel file. Keep the dataset inside the project's `data/` folder.

In [ ]:
DATA_PATH = Path("data/your_dataset.csv")  # Example: data/telco_churn.csv

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"File not found: {DATA_PATH}. Put your CSV/XLSX in the data folder "
        "and update DATA_PATH."
    )

if DATA_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(DATA_PATH)
elif DATA_PATH.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(DATA_PATH)
else:
    raise ValueError("Supported formats: CSV, XLSX, XLS")

print("Dataset shape:", df.shape)
display(df.head())

## 4. Initial data inspection

In [ ]:
display(df.info())
display(df.describe(include="all").T)
print("Duplicate rows:", df.duplicated().sum())
print("\\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

## 5. Basic data cleaning
This removes exact duplicate rows, trims whitespace in column names and text cells, and normalizes common blank strings to missing values. Review the changes before modeling.

In [ ]:
df_clean = df.copy()
df_clean.columns = df_clean.columns.astype(str).str.strip()
df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()]
df_clean = df_clean.drop_duplicates()

for col in df_clean.select_dtypes(include=["object", "string"]).columns:
    df_clean[col] = df_clean[col].astype("string").str.strip()
    df_clean[col] = df_clean[col].replace({"": pd.NA, "NA": pd.NA, "N/A": pd.NA, "null": pd.NA})

print("Before:", df.shape, "| After:", df_clean.shape)
display(df_clean.head())
display(df_clean.isna().sum().sort_values(ascending=False).to_frame("missing_after_cleaning"))

## 6. Exploratory Data Analysis (EDA)

In [ ]:
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))
print("Numeric columns:", df_clean.select_dtypes(include=np.number).columns.tolist())
print("Categorical columns:", df_clean.select_dtypes(exclude=np.number).columns.tolist())

numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    display(df_clean[numeric_cols].describe().T)
    df_clean[numeric_cols].hist(figsize=(14, 9), bins=25)
    plt.tight_layout()
    plt.show()

missing_pct = (df_clean.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
if not missing_pct.empty:
    missing_pct.plot(kind="bar", figsize=(12, 4), title="Missing values (%)")
    plt.ylabel("Percent")
    plt.tight_layout()
    plt.show()

if len(numeric_cols) >= 2:
    plt.figure(figsize=(10, 7))
    sns.heatmap(df_clean[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Numeric feature correlation")
    plt.tight_layout()
    plt.show()

## 7. Choose the prediction target
Set the exact column name. Select `classification` for categories (e.g. churn yes/no) or `regression` for a numeric quantity (e.g. sales amount).

In [ ]:
print("Available columns:", df_clean.columns.tolist())
TARGET_COLUMN = "CHANGE_ME"  # Example: "Churn"
TASK = "classification"      # Choose: "classification" or "regression"

if TARGET_COLUMN not in df_clean.columns:
    raise ValueError("Set TARGET_COLUMN to one of the available columns.")
if TASK not in ["classification", "regression"]:
    raise ValueError("TASK must be classification or regression.")

model_df = df_clean.dropna(subset=[TARGET_COLUMN]).copy()
X = model_df.drop(columns=[TARGET_COLUMN])
y = model_df[TARGET_COLUMN]

# Remove columns that contain no usable values
X = X.dropna(axis=1, how="all")
feature_columns = X.columns.astype(str).tolist()

if X.empty:
    raise ValueError("No feature columns remain after selecting the target.")
if TASK == "classification" and y.nunique() < 2:
    raise ValueError("Classification needs at least two target classes.")

print("Task:", TASK)
print("Features:", feature_columns)
print("Target values / summary:")
display(y.value_counts(dropna=False).head(20) if TASK == "classification" else y.describe())

## 8. Prepare preprocessing and split data
Numeric missing values are filled with the median; categorical missing values with the most frequent value. Categorical columns are one-hot encoded. The preprocessing is saved inside the model pipeline.

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

stratify_arg = None
if TASK == "classification":
    counts = y.value_counts()
    if len(counts) > 1 and counts.min() >= 2:
        stratify_arg = y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify_arg
)

if TASK == "classification":
    estimator = RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced"
    )
else:
    estimator = RandomForestRegressor(
        n_estimators=300, random_state=42
    )

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", estimator)
])

model.fit(X_train, y_train)
print("Model training complete.")

## 9. Evaluate the model

In [ ]:
predictions = model.predict(X_test)

if TASK == "classification":
    print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
    print(classification_report(y_test, predictions, zero_division=0))
    labels = model.named_steps["model"].classes_
    cm = confusion_matrix(y_test, predictions, labels=labels)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion matrix")
    plt.tight_layout()
    plt.show()
else:
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R²:", round(r2, 4))

results = pd.DataFrame({"actual": y_test, "predicted": predictions})
display(results.head(15))

## 10. Export model and metadata
The Streamlit app can load these files to predict on a new CSV/XLSX file. Keep `model.joblib` and `model_info.json` together.

In [ ]:
model_path = ARTIFACTS_DIR / "model.joblib"
info_path = ARTIFACTS_DIR / "model_info.json"

joblib.dump(model, model_path)

model_info = {
    "task": TASK,
    "target_column": str(TARGET_COLUMN),
    "feature_columns": feature_columns,
    "created_with": "scikit-learn Pipeline",
    "model_name": type(model.named_steps["model"]).__name__
}
info_path.write_text(json.dumps(model_info, indent=2), encoding="utf-8")

print("Saved:", model_path.resolve())
print("Saved:", info_path.resolve())

## 11. Optional: export cleaned dataset

In [ ]:
cleaned_path = PROJECT_DIR / "data" / "cleaned_dataset.csv"
cleaned_path.parent.mkdir(exist_ok=True)
df_clean.to_csv(cleaned_path, index=False)
print("Saved:", cleaned_path.resolve())

## 12. Next steps

1. Confirm model metrics and check whether the target column makes sense.
2. Test the exported model with unseen rows in the Streamlit app.
3. Add a README with dataset source, target definition, metrics and limitations.
4. Do not upload private datasets, API keys, `.env`, or confidential files to GitHub.